# Merged BERTopic - Analysis

Load the saved merged model and article assignments from `Merged_BERTopic_Build.ipynb` and inspect results.

**Prerequisite:** Run `Merged_BERTopic_Build.ipynb` first with `SAVE=True`.

Steps:
1. Load merged model + article assignments
2. Topic overview (counts, keywords)
3. Per-outlet topic distribution table
4. UMAP coloured by outlet
5. UMAP coloured by topic

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from bertopic import BERTopic
from IPython.display import display

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not find project root (.git)")

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUTS_DIR = PROJECT_ROOT / "1a_BERTopic" / "outputs"
MERGED_SAVE_DIR = OUTPUTS_DIR / "merged_model"

os.environ["NUMBA_CACHE_DIR"] = str(PROJECT_ROOT / ".numba_cache")
print(f"Outputs dir: {OUTPUTS_DIR}")

## 1. Load merged model and article assignments

In [ ]:
merged_model = BERTopic.load(MERGED_SAVE_DIR, embedding_model=EMBEDDING_MODEL)
topic_info = merged_model.get_topic_info()

n_topics = len(topic_info[topic_info["Topic"] != -1])
print(f"Loaded merged model: {n_topics} substantive topics")

assignments_path = OUTPUTS_DIR / "merged_article_topics.parquet"
df = pd.read_parquet(assignments_path)
print(f"Loaded article assignments: {len(df):,} articles")
print(df["source"].value_counts())

## 2. Topic overview

In [ ]:
topic_kw_map = dict(zip(topic_info["Topic"], topic_info["Name"].str.replace(r"^\d+_", "", regex=True)))

overview = (
    topic_info[topic_info["Topic"] != -1]
    .copy()
    .assign(Keywords=lambda d: d["Name"].str.replace(r"^\d+_", "", regex=True))
    [["Topic", "Count", "Keywords"]]
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)
overview.index += 1

print(f"Total topics: {len(overview)}")
display(overview)

## 3. Per-outlet topic distribution

Rows = topics, columns = outlets, values = article counts. `TOTAL` = sum across all outlets.

In [ ]:
SOURCE_ORDER = [
    "Tagesschau", "RT", "Antispiegel",
    "Tichys_Einblick", "Nius", "Compact", "Deutschlandkurier",
]
SOURCE_LABELS = {
    "Tagesschau": "Tagesschau", "RT": "RT DE", "Antispiegel": "Anti-Spiegel",
    "Tichys_Einblick": "Tichys", "Nius": "Nius",
    "Compact": "Compact", "Deutschlandkurier": "DK",
}

df_sub = df[df["merged_topic"] != -1].copy()
counts = (
    df_sub.groupby(["merged_topic", "source"]).size().unstack(fill_value=0)
)
present_cols = [c for c in SOURCE_ORDER if c in counts.columns]
counts = counts[present_cols]
counts.columns = [SOURCE_LABELS.get(c, c) for c in counts.columns]
counts["TOTAL"] = counts.sum(axis=1)
counts = counts.sort_values("TOTAL", ascending=False)
counts.insert(0, "Keywords", counts.index.map(topic_kw_map))

display(counts)

counts_path = OUTPUTS_DIR / "topic_outlet_counts.csv"
counts.to_csv(counts_path)
print(f"Saved: {counts_path}")

In [ ]:
# Per-outlet shares: each outlet's topic distribution sums to 100%
count_cols = [c for c in counts.columns if c not in ["Keywords", "TOTAL"]]
shares = counts[count_cols].div(counts[count_cols].sum(axis=0), axis=1).mul(100).round(1)
shares.insert(0, "Keywords", counts["Keywords"])
shares["TOTAL_count"] = counts["TOTAL"]

display(shares.head(30))

shares_path = OUTPUTS_DIR / "topic_outlet_shares.csv"
shares.to_csv(shares_path)
print(f"Saved: {shares_path}")

## 4. UMAP - coloured by outlet

Embed all articles and project to 2D. Each dot = one article, coloured by outlet.
Clusters that are dominated by one outlet show outlet-specific agenda areas.

In [ ]:
from sentence_transformers import SentenceTransformer
from umap import UMAP

print("Embedding articles ...")
encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = encoder.encode(df["doc_text"].tolist(), show_progress_bar=True, batch_size=256)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
print("Fitting 2D UMAP ...")
reducer = UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
umap_2d = reducer.fit_transform(embeddings)
df["umap_x"] = umap_2d[:, 0]
df["umap_y"] = umap_2d[:, 1]
print("Done.")

In [ ]:
OUTLET_COLORS = {
    "Tagesschau":        "#1f77b4",
    "RT":                "#d62728",
    "Antispiegel":       "#8c564b",
    "Tichys_Einblick":   "#ff7f0e",
    "Nius":              "#2ca02c",
    "Compact":           "#9467bd",
    "Deutschlandkurier": "#17becf",
}

fig, ax = plt.subplots(figsize=(14, 10))
for source in SOURCE_ORDER:
    mask = df["source"] == source
    ax.scatter(
        df.loc[mask, "umap_x"], df.loc[mask, "umap_y"],
        s=3, alpha=0.35, color=OUTLET_COLORS.get(source, "grey"),
        label=f"{SOURCE_LABELS.get(source, source)} (n={mask.sum():,})",
        rasterized=True,
    )

ax.legend(loc="upper right", markerscale=4, framealpha=0.8)
ax.set_title("Merged BERTopic — UMAP coloured by outlet", fontsize=14)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

fig.savefig(OUTPUTS_DIR / "umap_outlet_colored.pdf", dpi=150, bbox_inches="tight")
print(f"Saved: {OUTPUTS_DIR / 'umap_outlet_colored.pdf'}")
plt.show()

## 5. UMAP - coloured by topic (top N)

In [ ]:
TOP_N = 20

top_topics = (
    df[df["merged_topic"] != -1]
    .groupby("merged_topic").size()
    .sort_values(ascending=False).head(TOP_N).index.tolist()
)
cmap = plt.get_cmap("tab20", len(top_topics))
topic_color_map = {t: mcolors.to_hex(cmap(i)) for i, t in enumerate(top_topics)}

fig, ax = plt.subplots(figsize=(14, 10))

mask_other = ~df["merged_topic"].isin(top_topics)
ax.scatter(df.loc[mask_other, "umap_x"], df.loc[mask_other, "umap_y"],
           s=2, alpha=0.1, color="lightgrey", rasterized=True, label="Other / outlier")

for t in top_topics:
    mask = df["merged_topic"] == t
    kw = topic_kw_map.get(t, "")[:30]
    ax.scatter(df.loc[mask, "umap_x"], df.loc[mask, "umap_y"],
               s=4, alpha=0.5, color=topic_color_map[t],
               label=f"T{t}: {kw}", rasterized=True)

ax.legend(loc="upper right", markerscale=3, fontsize=7, framealpha=0.8)
ax.set_title(f"Merged BERTopic — Top {TOP_N} topics in UMAP", fontsize=14)
ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()

fig.savefig(OUTPUTS_DIR / "umap_topic_colored.pdf", dpi=150, bbox_inches="tight")
print(f"Saved: {OUTPUTS_DIR / 'umap_topic_colored.pdf'}")
plt.show()